In [85]:
import numpy as np
import pandas as pd
import os
import joblib
import json
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve, precision_recall_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [86]:
# ================= 配置参数 =================
DATA_DIR = "./benchmark_dataset_rpm_offset_100/"
OUTPUT_DIR = "./results/isolation_forest/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 模型参数
MODEL_CONFIG = {
    'n_estimators': 100,           # 树的数量
    'max_samples': 'auto',         # 每个树使用的样本数
    'contamination': 0.1,          # 异常比例（初步估计）
    'max_features': 1.0,           # 每个树使用的特征比例
    'bootstrap': False,            # 是否使用bootstrap采样
    'random_state': 42,            # 随机种子（保证可复现）
    'n_jobs': -1,                  # 使用所有CPU核心
    'verbose': 1                   # 打印训练进度
}

# 超参数搜索范围（用于验证集调优）
HPARAM_GRID = {
    'contamination': [0.01, 0.05, 0.08, 0.1, 0.12, 0.15, 0.2],
    'n_estimators': [50, 100, 150, 200],
    'max_samples': ['auto', 0.5, 0.7, 0.9]
}

# 评估阈值
THRESHOLD_METRIC = 'f1'  # 使用F1-score选择最优阈值

In [87]:

def load_benchmark_data(data_dir):    
    """
    加载所有基准数据集
    """
    print("加载数据...")
    
    # 加载训练集（仅正常数据）
    train_df = pd.read_parquet(os.path.join(data_dir, "train.parquet"))
    train_data = np.load(os.path.join(data_dir, "train_data.npy"))
    
    # 加载验证集（正常+合成异常）
    val_df = pd.read_parquet(os.path.join(data_dir, "validation_benchmark.parquet"))
    val_data = np.load(os.path.join(data_dir, "validation_data.npy"))
    
    # 加载测试集（正常+合成异常）
    test_df = pd.read_parquet(os.path.join(data_dir, "test_benchmark.parquet"))
    test_data = np.load(os.path.join(data_dir, "test_data.npy"))
    
    print(f"训练集: {train_data.shape}, 正常={np.sum(train_df['label']==0)}, 异常={np.sum(train_df['label']==1)}")
    print(f"验证集: {val_data.shape}, 正常={np.sum(val_df['label']==0)}, 异常={np.sum(val_df['label']==1)}")
    print(f"测试集: {test_data.shape}, 正常={np.sum(test_df['label']==0)}, 异常={np.sum(test_df['label']==1)}")
    
    return {
        'train': {'data': train_data, 'df': train_df},
        'validation': {'data': val_data, 'df': val_df},
        'test': {'data': test_data, 'df': test_df}
    }

In [88]:
# ================= 主执行流程 =================
print("=" * 60)
print("ISOLATION FOREST BASELINE")
print("=" * 60)
    
# 1. 加载数据
datasets = load_benchmark_data(DATA_DIR)
train_data = datasets['train']['data']   
train_df = datasets['train']['df']
val_data = datasets['validation']['data']
val_df = datasets['validation']['df']
test_data = datasets['test']['data']
test_df = datasets['test']['df']

ISOLATION FOREST BASELINE
加载数据...
训练集: (18254, 60, 1), 正常=18254, 异常=0
验证集: (6085, 60, 1), 正常=6055, 异常=30
测试集: (6085, 60, 1), 正常=6055, 异常=30


In [89]:
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)



# We use only ENGINE_RPM ().
#
# Each sample has 60 timesteps.
#
# Input:
#   train_data: (N, 60, 27)
#
# Output:
#   train_features: (N, 60)
#
# This is the same representation as your Flat=True path.

rpm_idx = 0



# ============================================================
# 3. Sanity check features
# ============================================================

def extract_rpm_features(data):
    """
    Extract RPM time series.

    Input:
        data: (N, 60, 27)

    Output:
        features: (N, 60)
    """
    return np.asarray(
        data[:, :, rpm_idx],
        dtype=np.float64
    )


X_train = extract_rpm_features(train_data)
X_val = extract_rpm_features(val_data)

print("RPM index:", rpm_idx)
print("X_train shape:", X_train.shape)
print("X_train dtype:", X_train.dtype)
print("X_val shape:", X_val.shape)
print("X_val dtype:", X_val.dtype)


print("\n" + "=" * 60)
print("Training Isolation Forest")
print("=" * 60)

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)

# Use X_train, NOT train_features
iso_forest.fit(X_train)

print("Isolation Forest training completed.")


# ============================================================
# 5. Generate anomaly scores
# ============================================================

print("\n" + "=" * 60)
print("Generating anomaly scores")
print("=" * 60)

# Lower = more anomalous
val_scores = iso_forest.score_samples(X_val)

print("Score range:")
print("  min:", val_scores.min())
print("  max:", val_scores.max())


# ============================================================
# 6. Rank-based threshold selection
# ============================================================

print("\n" + "=" * 60)
print("Selecting anomaly threshold")
print("=" * 60)

# We have 30 synthetic anomalies in validation.
#
# From the previous experiment:
#
#   top 30:
#       TP=29, FP=1
#
#   top 31:
#       TP=30, FP=1
#
# Therefore top-31 gives:
#
#       recall    = 100%
#       precision = 96.77%
#
# for the +100 RPM sanity test.

ranked_idx = np.argsort(val_scores)

k = 31

threshold = val_scores[ranked_idx[k - 1]]

print("Number of samples selected:", k)
print("Selected threshold:", threshold)


# ============================================================
# 7. Evaluate Isolation Forest
# ============================================================

print("\n" + "=" * 60)
print("Isolation Forest evaluation")
print("=" * 60)

# Lower score = more anomalous
pred = (val_scores <= threshold).astype(int)

precision = precision_score(
    val_labels,
    pred,
    zero_division=0
)

recall = recall_score(
    val_labels,
    pred,
    zero_division=0
)

f1 = f1_score(
    val_labels,
    pred,
    zero_division=0
)

tp = ((pred == 1) & (val_labels == 1)).sum()
fp = ((pred == 1) & (val_labels == 0)).sum()
fn = ((pred == 0) & (val_labels == 1)).sum()
tn = ((pred == 0) & (val_labels == 0)).sum()

print(f"Threshold : {threshold:.6f}")
print(f"TP        : {tp}")
print(f"FP        : {fp}")
print(f"FN        : {fn}")
print(f"TN        : {tn}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1        : {f1:.4f}")


# ============================================================
# 8. Inspect false positives
# ============================================================

print("\n" + "=" * 60)
print("False positive analysis")
print("=" * 60)

fp_idx = np.where(
    (pred == 1) & (val_labels == 0)
)[0]

print("Number of false positives:", len(fp_idx))

for idx in fp_idx[:50]:

    rpm = val_data[idx, :, rpm_idx]

    print(
        f"idx={idx:4d} "
        f"mean={rpm.mean():8.3f} "
        f"max={rpm.max():8.3f} "
        f"min={rpm.min():8.3f} "
        f"score={val_scores[idx]:10.6f}"
    )


# ============================================================
# 9. RPM-max baseline
# ============================================================

print("\n" + "=" * 60)
print("RPM max baseline")
print("=" * 60)

train_rpm_max = train_data[:, :, rpm_idx].max(axis=1)
val_rpm_max = val_data[:, :, rpm_idx].max(axis=1)


print("\nTrain RPM max percentiles:")
print(
    np.percentile(
        train_rpm_max,
        [90, 95, 99, 99.5, 99.9, 99.95, 99.99, 100]
    )
)


print("\nSynthetic anomaly RPM max percentiles:")
print(
    np.percentile(
        val_rpm_max[val_labels == 1],
        [0, 5, 25, 50, 75, 95, 100]
    )
)


# For the +100 RPM sanity test, RPM max > 9
# was previously found to perfectly separate
# synthetic anomalies from the rest of validation data.

rpm_threshold = 9.0

rpm_pred = (
    val_rpm_max > rpm_threshold
).astype(int)

rpm_precision = precision_score(
    val_labels,
    rpm_pred,
    zero_division=0
)

rpm_recall = recall_score(
    val_labels,
    rpm_pred,
    zero_division=0
)

rpm_f1 = f1_score(
    val_labels,
    rpm_pred,
    zero_division=0
)

rpm_tp = ((rpm_pred == 1) & (val_labels == 1)).sum()
rpm_fp = ((rpm_pred == 1) & (val_labels == 0)).sum()
rpm_fn = ((rpm_pred == 0) & (val_labels == 1)).sum()

print("\nRPM max baseline:")
print(f"Threshold : {rpm_threshold:.1f}")
print(f"TP        : {rpm_tp}")
print(f"FP        : {rpm_fp}")
print(f"FN        : {rpm_fn}")
print(f"Precision : {rpm_precision:.4f}")
print(f"Recall    : {rpm_recall:.4f}")
print(f"F1        : {rpm_f1:.4f}")

RPM index: 0
X_train shape: (18254, 60)
X_train dtype: float64
X_val shape: (6085, 60)
X_val dtype: float64

Training Isolation Forest
Isolation Forest training completed.

Generating anomaly scores
Score range:
  min: -0.7955781670529618
  max: -0.3402449900479585

Selecting anomaly threshold
Number of samples selected: 31
Selected threshold: -0.7934279687532505

Isolation Forest evaluation
Threshold : -0.793428
TP        : 30
FP        : 1
FN        : 0
TN        : 6054
Precision : 0.9677
Recall    : 1.0000
F1        : 0.9836

False positive analysis
Number of false positives: 1
idx=1228 mean=   7.186 max=   7.747 min=   6.471 score= -0.793428

RPM max baseline

Train RPM max percentiles:
[-0.06317824  2.75669426  6.31103893  6.89115518  7.74662833  7.94896048
  8.20923011  8.20923011]

Synthetic anomaly RPM max percentiles:
[ 99.70552087  99.70552087  99.73358311  99.81776983  99.86368986
 103.1733335  108.20923011]

RPM max baseline:
Threshold : 9.0
TP        : 30
FP        : 0
FN 